# HST Corpus 80m Analysis

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 21.02.2025
- Date of Last Modification: 10.06.2025 (by Sebastian Birk; sb75@sanger.ac.uk)

# 1. Setup

## 1.1 Load Libraries 

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../../utils")

In [3]:
import logging
import os
import warnings
import pickle
import random
from datetime import datetime

import anndata as ad
import torch
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scib_metrics
import seaborn as sns
import squidpy as sq
from datasets import load_from_disk
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split

from app.infer import infer
from app.train import train
from nichejepa.datasets import prepare_dataset
from nichejepa.utils.config import create_params_from_YAML_wandb_config
from nichejepa.utils.distributed import init_distributed
from nichejepa.utils.evaluation import clustering_metrics, classification_metrics

from analysis_utils import *

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_text from `anndata` is deprecated. Import anndata.io.read_text instead.
  warnings.warn(msg, FutureWarning)


## 1.2 Load Config/Params
All setup will be done in the config file. Please edit that file to continue.

In [4]:
# Set params
dataset = 'hst_corpus_80m'
tokenizer = 'cell_graph'
# load_timestamp = '29032025_080927_875' # gt_tiny
# load_timestamp = '03042025_224852_256' # gt_small
# load_timestamp = '27032025_163938_061' # gt_base
# load_timestamp = '03042025_224013_784' # gt_large
# load_timestamp = None
load_timestamp = '18062025_082526_412'
model_type = 'test'
read_checkpoint = 'nichejepa-ep7_0.pth.tar' # 'nichejepa-ep1_6.pth.tar'

In [5]:
model_config_folder_path = '/nfs/team361/sb75/nichejepa/configs/model' # '/nfs/team361/sb75/nichejepa/configs/model' # '/lustre/scratch126/cellgen/team361/sb75/nichejepa/configs/model'
artifacts_folder_path = '/nfs/team361/sb75/nichejepa-reproducibility/artifacts' # '/nfs/team361/sb75/nichejepa-reproducibility/artifacts' # '/nfs/team361/sb75/nichejepa-reproducibility/artifacts' # '/lustre/scratch126/cellgen/lotfollahi/sb75/nichejepa-reproducibility/artifacts' # '/nfs/team361/sb75/nichejepa-reproducibility/artifacts'
datasets_folder_path = '/nfs/team361/sb75/DATASETS' # '/lustre/scratch126/cellgen/team361/DATASETS'
token_dict_path = f'{datasets_folder_path}/tokenizer/token_dictionary_homo_sapiens.pkl'

In [6]:
# Set up logger
rank=0
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

In [7]:
# Load config
if not load_timestamp:
    params = create_params_from_YAML_wandb_config(f'{model_config_folder_path}/{dataset}_{model_type}.yaml',logger)
else:
    params = create_params_from_YAML_wandb_config(f'{artifacts_folder_path}/{dataset}/{load_timestamp}/params.yaml',logger)

if read_checkpoint:
    params['state']['read_checkpoint'] = read_checkpoint
if load_timestamp:
    params['state']['load_checkpoint'] = True

params

INFO:root:Loaded parameters from YAML file.


{'data': {'batch_size': 144,
  'dataset_name': 'hst_corpus_80m',
  'n_segments': 11,
  'num_workers': 48,
  'pin_memory': True,
  'precomputed_epoch_n_nonzero_tokens': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_n_nonzero_tokens_30.pkl',
  'precomputed_epoch_splits': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10_precomputed_epoch_split_30.pkl',
  'random_state': 0,
  'raw_data_folder_path': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/hst_corpus_80m',
  'sample_subset': False,
  'sampling_strategy': None,
  'seq_len_cell': 256,
  'seq_len_neighborhood': 2560,
  'split': 0,
  'stratify': False,
  'token_dict_folder_path': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/token_dictionary_homo_sapiens.pkl',
  'tokenized_data_folder_path': '/lustre/scratch126/cellgen/lotfollahi/DATASETS/gold/cell-graph-tok

In [8]:
# Define folder path for saving and loading artifacts
if not load_timestamp:
    current_timestamp = (
        datetime.now().strftime("%d%m%Y_%H%M%S") +
        f"_{datetime.now().microsecond // 1000:03d}")
    folder_path = os.path.join(artifacts_folder_path,
                               params['data']['dataset_name'],
                               current_timestamp)
else:
    folder_path = f'{artifacts_folder_path}/{dataset}/{load_timestamp}'
    
folder_path

'/nfs/team361/sb75/nichejepa-reproducibility/artifacts/hst_corpus_80m/18062025_082526_412'

## 1.3 Run Notebook Setup

In [9]:
os.makedirs(f'{folder_path}/figures', exist_ok=True)

In [10]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [11]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [12]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [13]:
sc.settings.figdir = f'{folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 4),
    facecolor='white',
    fontsize=10
)

## 1.4 Define Functions

In [14]:
def run_inference(dataset_name,
                  params,
                  read_checkpoint,
                  obs_cols,
                  rerun_inference=False,
                  write_to_disk=True):
    # Load inference results if they exist
    if os.path.exists(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad') and not rerun_inference:
        adata = sc.read_h5ad(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    else:
        # Load data
        _, _, test_dataset = prepare_dataset(params)
        test_dataset
        
        # Run model inference
        adata = infer(params,
                      test_dataset,
                      load_folder_path=folder_path,
                      dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
                      obs_cols=obs_cols,
                      agg_type='avg',
                      masked_tokens=None,
                      agg_excluded_tokens=None,
                      cell_gene_ids=[],
                      neighborhood_gene_ids=[])

    # Extract batch from cell ID
    adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]
    
    # Remove missing ground truth
    for obs_col in obs_cols:
        adata = adata[~adata.obs[obs_col].isna()]

    if write_to_disk:
        # Save adata
        adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

    return adata

In [15]:
def compute_embedding_similarity(adata,
                                 emb_keys=['cell', 'neighborhood'],
                                 label='niche_type',
                                 write_to_disk=True):
    for emb_key in emb_keys:
        similarity_stats = compute_similarity_stats(
            adata,
            n_samples=None,
            type_key=label,
            embedding_key=f'{emb_key}_emb_layer_{layer}')

        avg_cos_sim = np.mean(similarity_stats["Mean Similarity"])
    
        cosine_df = pd.DataFrame({
            'metric': [f'avg_{label}_cos_sim'],
            'value': [avg_cos_sim]
        })
        print(f'Average {label} cosine similarity is {avg_cos_sim}.')
        cosine_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_avg_{label}_cos_sim.csv',
                         index=False)
        
        plot_similarity_heatmaps(
            similarity_stats,
            save_file_path=f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{emb_key}_emb_{label}_cos_sim.svg')

In [16]:
def compute_neighborhood_umap(adata,
                              write_to_disk=True):
    if 'neighborhood_connectivities' not in adata.obsp.keys():
        # Compute UMAP representation for (aggregated) neighborhood gene embeddings
        sc.pp.neighbors(adata,
                        n_neighbors=15,
                        use_rep=f'neighborhood_emb_layer_{layer}',
                        key_added='neighborhood')
    if 'neighborhood_umap' not in adata.obsm.keys():
        sc.tl.umap(adata,
                   neighbors_key='neighborhood',
                   key_added='neighborhood_umap')
    
        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    
    if 'cell_connectivities' not in adata.obsp.keys():
        # Compute UMAP representation for (aggregated) cell gene embeddings
        sc.pp.neighbors(adata,
                        n_neighbors=15,
                        use_rep=f'cell_emb_layer_{layer}',
                        key_added='cell')
    if 'cell_umap' not in adata.obsm.keys():
        sc.tl.umap(adata,
                   neighbors_key='cell',
                   key_added='cell_umap')

        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

    return adata

In [17]:
def compute_latent_clusters(adata,
                            emb_keys=['cell', 'neighborhood'],
                            obs_col='niche_type',
                            latent_leiden_resolution_cell=1.0,
                            latent_leiden_resolution_neighborhood=0.4,
                            write_to_disk=True):
    for emb_key in emb_keys:
        latent_leiden_resolutions = []
        if emb_key == 'cell':
            latent_leiden_resolution = latent_leiden_resolution_cell
        elif emb_key == 'neighborhood':
            latent_leiden_resolution = latent_leiden_resolution_neighborhood
        adata.obsm['X_umap'] = adata.obsm[f'{emb_key}_umap']
        # Perform clustering until number of ground truth niches is matched
        i = 0
        n_unique_vals = len(adata.obs[obs_col].value_counts() < 100)
        print(f'Target number of unique values: {n_unique_vals}.')
        n_latent_clusters = 0
        while (n_latent_clusters != n_unique_vals):
            if latent_leiden_resolution in latent_leiden_resolutions:
                latent_leiden_resolution += random.uniform(0.0001, 0.001)
            latent_leiden_resolutions.append(latent_leiden_resolution)
            print(f'Leiden resolution: {latent_leiden_resolution}.')
            latent_cluster_key = f'{emb_key}_emb_leiden_res{str(latent_leiden_resolution).replace(".", "_")}'
            i += 1
            sc.tl.leiden(adata,
                         neighbors_key=emb_key,
                         key_added=latent_cluster_key,
                         resolution=latent_leiden_resolution)
        
            n_latent_clusters = len(adata.obs[latent_cluster_key].value_counts() < 100)
            print(f"Number of latent clusters {n_latent_clusters}.")
            if n_latent_clusters == n_unique_vals:
                break
            elif n_latent_clusters < (n_unique_vals - 8):
                latent_leiden_resolution += 0.1
            elif n_latent_clusters < (n_unique_vals - 5):
                latent_leiden_resolution += 0.05
            elif n_latent_clusters < (n_unique_vals - 3):
                latent_leiden_resolution += 0.02
            elif n_latent_clusters < n_unique_vals:
                latent_leiden_resolution += 0.001
            elif n_latent_clusters > (n_unique_vals + 8):
                latent_leiden_resolution -= 0.1
            elif n_latent_clusters > (n_unique_vals + 5):
                latent_leiden_resolution -= 0.05
            elif n_latent_clusters > (n_unique_vals + 3):
                latent_leiden_resolution -= 0.02
            else:
                latent_leiden_resolution -= 0.001
        
        # Compute metrics and store to disk
        labels = adata.obs[obs_col]
        labels_pred = adata.obs[latent_cluster_key]
        nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
        ari = adjusted_rand_score(labels, labels_pred)
        print(f'NMI is {nmi}. ARI is {ari}.')
        metrics_df = pd.DataFrame({
            'metric': ['NMI', 'ARI'],
            'value': [nmi, ari]
        })
        metrics_df.to_csv(f'{folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_metrics.csv',
                          index=False)
        
        # Plot UMAPs
        os.makedirs(f'{folder_path}/figures', exist_ok=True)
        labels = ['batch', obs_col, latent_cluster_key]
        # Plot (aggregated) neighborhood gene embeddings, colored by labels
        for label in labels:
            if len(np.unique(adata.obs[label])) > 1:
                sc.pl.umap(adata,
                           neighbors_key=emb_key,
                           title=(emb_key + "_emb_" + label if emb_key not in label else label),
                           color=label,
                           palette=(custom_palette if label == latent_cluster_key else None),
                           show=False)
                plt.savefig(f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}{"_" + emb_key + "_emb" if emb_key not in label else ""}_{label}_umap.svg',
                            dpi=300,
                            bbox_inches='tight')
        
        # Plot niches in tissue
        batch_ids = adata.obs['batch'].unique().tolist()
        for i, batch_id in enumerate(batch_ids):
            adata_batch = adata[adata.obs['batch'] == batch_id]
            sc.pl.spatial(adata_batch,
                          color=latent_cluster_key,
                          palette=custom_palette,
                          spot_size=spot_sizes[i],
                          save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg')
            os.rename(f'{folder_path}/figures/show{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg',
                      f'{folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'.replace("show", ""))
    
        if write_to_disk:
            # Save adata
            adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    
    return adata

# 2. Analysis

In [18]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
params['data']['batch_size'] = 128

In [19]:
params['data']['precomputed_n_nonzero_tokens'] = '/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/hst_corpus_80m_32_None_None_None_None_None_shifted_log_knn_10_n_nonzero_tokens_train.pkl'

In [ ]:
# Load data
train_dataset, _, test_dataset = prepare_dataset(params)
train_dataset

In [ ]:
devices=['cuda:0']
world_size=1
os.environ['CUDA_VISIBLE_DEVICES'] = str(devices[rank].split(':')[-1])

logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

world_size, rank = init_distributed(rank_and_world_size=(rank, world_size), port=random.randint(40000, 50000))
logger.info(f'Running... (rank: {rank}/{world_size})')

## 2.1 Benchmarking Test Sets

### 2.1.1 xhs49-1b_1p (Xenium Human Skin Melanoma)

#### 2.1.1.1 Define Config

In [20]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '49_batch0']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [21]:
# Set dataset params
dataset_name = 'xhs49-1b_1p'
niche_key = 'niche_name'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [20] * len(params['data']['test_batch_ids'])

#### 2.1.1.2 Run Model Inference

In [22]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[niche_key],
                      write_to_disk=True)

INFO:root:EncoderMultiMaskWrapper(
  (backbone): GeneTransformerCountEncoder(
    (token_embed): Embedding(23407, 384, padding_idx=0)
    (seg_embed): Embedding(12, 384, padding_idx=0)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=384, out_features=1152, bias=True)
          (attn_drop): Dropout(p=0.0, inplace=False)
          (proj): Linear(in_features=384, out_features=384, bias=True)
          (proj_drop): Dropout(p=0.0, inplace=False)
        )
        (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
        (mlp): MLP(
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (act): GELU(approximate='none')
          (fc2): Linear(in_features=1536, out_features=384, bias=True)
          (drop): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (norm): LayerNorm((384,), eps=1e-06, elementwise_affine=

['encoder', 'predictor', 'target_encoder', 'opt', 'scaler', 'epoch', 'zero_epoch_tracking', 'loss', 'batch_size', 'world_size', 'lr', 'iter_number']


0it [00:00, ?it/s]/nfs/team361/sb75/nichejepa/src/app/infer.py:357: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(dtype=torch.bfloat16,
/software/cellgen/team361/sb75/envs/nichejepa/lib/python3.10/contextlib.py:103: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
854it [25:57,  1.82s/it]
/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


Loading metadata AnnDatas...
Loading datasets: ['49'].
Loading AnnData objects from /lustre/scratch126/cellgen/lotfollahi/DATASETS/silver/hst_corpus_80m/xhs49-1b_1p.


/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/anndata/_core/anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
... storing 'batch' as categorical


#### 2.1.1.3 Plot Embedding Similarity Stats

In [23]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'],
                             label=niche_key,
                             write_to_disk=True)

                   Class 1                   Class 2  Mean Similarity  \
0                 Epidemis                  Epidemis         0.883620   
1                 Epidemis        Superficial dermis         0.886346   
2                 Epidemis       Melanoma+infiltrate         0.914592   
3                 Epidemis                  Melanoma         0.913167   
4                 Epidemis     Immune infiltrate Thi         0.897042   
..                     ...                       ...              ...   
59  Melanoma+blood vessels                  Melanoma         0.968731   
60  Melanoma+blood vessels     Immune infiltrate Thi         0.955601   
61  Melanoma+blood vessels  Immune infiltrate Mac hi         0.957931   
62  Melanoma+blood vessels     Macrophage_fat_region         0.940411   
63  Melanoma+blood vessels    Melanoma+blood vessels         0.976366   

    Variance Similarity  
0              0.008875  
1              0.008974  
2              0.003064  
3              0.00

#### 2.1.1.4 Identify Niches & Cell Types

In [ ]:
adata = compute_neighborhood_umap(adata,
                                  write_to_disk=True)

adata = compute_latent_clusters(adata,
                                emb_keys=['neighborhood'],
                                obs_col=niche_key,
                                latent_leiden_resolution_cell=latent_leiden_resolution_cell,
                                latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
                                write_to_disk=True)

Target number of unique values: 8.
Leiden resolution: 0.4.


/tmp/ipykernel_2912826/1337707055.py:26: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata,


Number of latent clusters 10.
Leiden resolution: 0.399.


### 2.1.2 xhs1000-39b_1p (Xenium Human Skin Eczema)

#### 2.1.2.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '1000_batch11',
    '1000_batch19',
    '1000_batch32']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Set dataset params
dataset_name = 'xhs1000-39b_1p'
niche_key = 'niche_type'
batch_key = 'sample'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [20] * len(params['data']['test_batch_ids'])

#### 2.1.2.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[niche_key],
                      write_to_disk=True)

#### 2.1.2.3 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'],
                             label=niche_key,
                             write_to_disk=True)

#### 2.1.2.4 Identify Niches & Cell Types

In [ ]:
adata = compute_neighborhood_umap(adata,
                                  write_to_disk=True)

adata = compute_latent_clusters(adata,
                                emb_keys=['neighborhood'],
                                obs_col=niche_key,
                                latent_leiden_resolution_cell=latent_leiden_resolution_cell,
                                latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
                                write_to_disk=True)

### 2.1.3 xhk1007-41b_1p (Xenium Human Kidney RCC)

#### 2.1.3.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
     '1007_batch1',
     '1007_batch14'
]
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Set dataset params
dataset_name = 'xhk1007-41b_1p'
cell_type_key = 'level_1_cell_type'
batch_key = 'batch'
latent_leiden_resolution_cell = 0.75
latent_leiden_resolution_neighborhood = 0.2
spot_sizes = [30] * len(params['data']['test_batch_ids'])

#### 2.1.3.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[cell_type_key],
                      write_to_disk=True)

#### 2.1.3.3 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['cell'],
                             label=cell_type_key,
                             write_to_disk=True)

#### 2.1.3.4 Identify Niches & Cell Types

In [ ]:
adata = compute_neighborhood_umap(adata,
                                  write_to_disk=True)

adata = compute_latent_clusters(adata,
                                emb_keys=['cell'],
                                obs_col=cell_type_key,
                                latent_leiden_resolution_cell=latent_leiden_resolution_cell,
                                latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
                                write_to_disk=True)

### 2.1.4 xhb1002-75b_3p (Xenium Human Brain GBM)

#### 2.1.4.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '1002_batch3',
    '1002_batch15'
    '1002_batch27']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Set dataset params
dataset_name = 'xhb1002-75b_3p'
cell_type_key = 'leiden_log1pnorm_res-1'
batch_key = 'batch'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [30] * len(params['data']['test_batch_ids'])

#### 2.1.4.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[cell_type_key],
                      write_to_disk=True)

#### 2.1.4.3 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['cell'],
                             label=cell_type_key,
                             write_to_disk=True)

#### 2.1.4.4 Identify Niches & Cell Types

In [ ]:
adata = compute_neighborhood_umap(adata,
                                  write_to_disk=True)

adata = compute_latent_clusters(adata,
                                emb_keys=['cell'],
                                obs_col=cell_type_key,
                                latent_leiden_resolution_cell=latent_leiden_resolution_cell,
                                latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
                                write_to_disk=True)

### 2.1.5 chl59-8b_1p (NanoString CosMx Human Lung NSCLC)

#### 2.1.5.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '59_batch0',
    #'59_batch3', # unannotated
    '59_batch6',
    '59_batch7',
    #'59_batch2', # unannotated
    '59_batch4',
    '59_batch1',
    '59_batch5']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Set dataset params
dataset_name = 'chl59-8b_1p'
niche_key = 'niche'
batch_key = 'batch'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.6
spot_sizes = [80] * len(params['data']['test_batch_ids'])

#### 2.1.5.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[niche_key],
                      write_to_disk=True)

#### 2.1.5.3 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'],
                             label=niche_key,
                             write_to_disk=True)

#### 2.1.5.4 Identify Niches & Cell Types

In [ ]:
adata = compute_neighborhood_umap(adata,
                                  write_to_disk=True)

adata = compute_latent_clusters(adata,
                                emb_keys=['neighborhood'],
                                obs_col=niche_key,
                                latent_leiden_resolution_cell=latent_leiden_resolution_cell,
                                latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
                                write_to_disk=True)

### 2.1.6 chr78-11b_1p (ISS Cartana Human Reproductive Tract)

#### 2.1.6.1 Define Config

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
    '78_batch3',
    '78_batch10',
    '78_batch1',
    '78_batch0',
    '78_batch2',
    '78_batch6',
    '78_batch7',
    '78_batch9',
    '78_batch8',
    '78_batch4',
    '78_batch5']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Set dataset params
dataset_name = 'chr78-11b_1p'
niche_key = 'broad_anatomy'
batch_key = 'batch'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [1920 * 0.093] * len(params['data']['test_batch_ids'])

#### 2.1.6.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[niche_key],
                      write_to_disk=True)

#### 2.1.6.3 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'],
                             label=niche_key,
                             write_to_disk=True)

#### 2.1.6.4 Identify Niches & Cell Types

In [ ]:
adata = compute_neighborhood_umap(adata,
                                  write_to_disk=True)

adata = compute_latent_clusters(adata,
                                emb_keys=['neighborhood'],
                                obs_col=niche_key,
                                latent_leiden_resolution_cell=latent_leiden_resolution_cell,
                                latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
                                write_to_disk=True)

## 2.2 Full Corpus

### 2.2.1 Run Model Inference

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

spv_dict = {
    k: v for k, v in token_dict.items() if k.startswith('spv_')}
spv_start_idx = min(spv_dict.values())
max_cls_tokens = sum(1 for key in token_dict if "cls" in key)
spv_idx_subtract = spv_start_idx - 2 - max_cls_tokens

In [ ]:
# Load dataset
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = False
params['data']['val_batch_ids'] = False
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

dataset, _, _ = prepare_dataset(params)
dataset

In [ ]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
#latent_leiden_resolution = 0.5 # 0.1, 0.5
#latent_cluster_key = f'leiden_res{str(latent_leiden_resolution).replace(".", "_")}'

In [ ]:
# Subset dataset
step_size = 500000
steps = int(np.ceil(len(dataset)/step_size))

dataset_subsets = []

for i in range(steps):
    if ((i+1)*step_size) > len(dataset):
        end_step = len(dataset)
    else:
        end_step = (i+1)*step_size
    dataset_subsets.append(dataset.select(np.arange(i*step_size, end_step)))

del dataset

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i > 4:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 5:
        continue
    if i > 9:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 10:
        continue
    if i > 14:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 15:
        continue
    if i > 19:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 20:
        continue
    if i > 24:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 25:
        continue
    if i > 29:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 30:
        continue
    if i > 34:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 35:
        continue
    if i > 39:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 40:
        continue
    if i > 44:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 45:
        continue
    if i > 49:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 50:
        continue
    if i > 54:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 55:
        continue
    if i > 59:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 60:
        continue
    if i > 64:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 65:
        continue
    if i > 69:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 70:
        continue
    if i > 74:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 75:
        continue
    if i > 79:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 80:
        continue
    if i > 84:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 85:
        continue
    if i > 89:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 90:
        continue
    if i > 94:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 95:
        continue
    if i > 99:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 100:
        continue
    if i > 104:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 155:
        continue
    if i > 159:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 (with 2M step size) need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 160:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[int(batch_value[0] + spv_idx_subtract)] for batch_value in dataset_subset['batch_value']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i != 6:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 7:
        continue
    if i > 12:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 13:
        continue
    if i > 18:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 19:
        continue
    if i > 25:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i != 26:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Get batches for current dataset subset
# Iteration 6 and 26 need 1TB RAM
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 27:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['assay', 'tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{step_size}_{i}.h5ad')
    del adata

### 2.2.2 Combine Data

In [23]:
load_timestamp = '11062025_183320_055'
step_size = '500000'
read_checkpoint = 'ep10_0'
#emb_key = 'neighborhood_emb_layer_6'

In [24]:
adata_list = []

for i in range(161):
    if i in [107, 138, 139, 140]: # 138 length, 
        continue
    adata = sc.read_h5ad(f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/adata_{read_checkpoint}_{step_size}_{i}.h5ad')
    adata_list.append(adata)
    del adata

adata_full = sc.concat(adata_list, join='outer', index_unique=None)

In [30]:
adata_full.obs[['dataset', 'batch', 'cell']] = [s.split('_') for s in adata_full.obs.index]
adata_full.obs['dataset_batch'] = adata_full.obs['dataset'].astype(str) + adata_full.obs['batch'].astype(str)
adata_full.obs['cell'] = adata_full.obs['cell'].astype(int)

In [26]:
adata_full.write(f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/adata_ep10_0_complete.h5ad')

In [27]:
sample_frac = 0.1
random_state = 42

In [29]:
adata_full.obs.head(1)

,assay,tissue,gene_panel_size
cell_id,,,
23_batch0_0,xenium,brain,466


In [31]:
# Perform stratified subsampling based on batch
sampled_df = adata_full.obs.groupby(['dataset', 'batch'], group_keys=False).apply(
        lambda x: x.sample(frac=sample_frac, replace=False, random_state=random_state))
adata_full = adata_full[sampled_df.index]
del(sampled_df)

/tmp/ipykernel_1470236/1907514910.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_df = adata_full.obs.groupby(['dataset', 'batch'], group_keys=False).apply(


In [32]:
n_layer =12 
n_pcs = 50

In [ ]:
# Define color palette
custom_palette_300 = {
    "0": "#5E67F7",
    "1": "#F54349",
    "2": "#D28E59",
    "3": "#920828",
    "4": "#07E10B",
    "5": "#13C35C",
    "6": "#91C83F",
    "7": "#61C7CC",
    "8": "#175DA6",
    "9": "#278729",
    "10": "#9A4B9B",
    "11": "#3516AA",
    "12": "#F223BF",
    "13": "#459A8C",
    "14": "#DADD17",
    "15": "#EF7E60",
    "16": "#5A58D5",
    "17": "#C22A30",
    "18": "#E96B13",
    "19": "#00DABE",
    "20": "#4DD8C1",
    "21": "#73A341",
    "22": "#3FD824",
    "23": "#9A0EA1",
    "24": "#338B6F",
    "25": "#D2C71E",
    "26": "#D660CF",
    "27": "#5346EE",
    "28": "#583581",
    "29": "#FC0F7F",
    "30": "#CC3335",
    "31": "#63CB9F",
    "32": "#88582B",
    "33": "#46FDF2",
    "34": "#DB8042",
    "35": "#0BDC84",
    "36": "#A5BD18",
    "37": "#F97487",
    "38": "#6F5ADA",
    "39": "#131591",
    "40": "#6EFD5F",
    "41": "#66AE4A",
    "42": "#A14E9D",
    "43": "#13B305",
    "44": "#E333E9",
    "45": "#46AAEE",
    "46": "#09868F",
    "47": "#CDE363",
    "48": "#9F07F3",
    "49": "#AB3324",
    "50": "#6647D1",
    "51": "#691E84",
    "52": "#8BB61B",
    "53": "#61FAAE",
    "54": "#E9A034",
    "55": "#E526AC",
    "56": "#1C871F",
    "57": "#3F8F73",
    "58": "#1DEBF4",
    "59": "#F93802",
    "60": "#9B4E21",
    "61": "#546BBB",
    "62": "#6C40FE",
    "63": "#76DD67",
    "64": "#F549F5",
    "65": "#B9062B",
    "66": "#C99961",
    "67": "#B64489",
    "68": "#21CE92",
    "69": "#10FCD1",
    "70": "#080ADF",
    "71": "#917816",
    "72": "#1082C9",
    "73": "#3A867D",
    "74": "#68F8CD",
    "75": "#E7E24A",
    "76": "#6A1B84",
    "77": "#F69054",
    "78": "#6E79F8",
    "79": "#ED3145",
    "80": "#55F474",
    "81": "#93B74F",
    "82": "#C96CF1",
    "83": "#8B083E",
    "84": "#3A6285",
    "85": "#069839",
    "86": "#018BB5",
    "87": "#23C8DE",
    "88": "#3E1AC0",
    "89": "#F97C95",
    "90": "#CCF55C",
    "91": "#B8506E",
    "92": "#C1BE16",
    "93": "#1AEB78",
    "94": "#78AE2E",
    "95": "#479067",
    "96": "#645FC3",
    "97": "#6DFD31",
    "98": "#A12D00",
    "99": "#E560DB",
    "100": "#DD7231",
    "101": "#020486",
    "102": "#E706AA",
    "103": "#29A5FD",
    "104": "#57FED0",
    "105": "#DCC95E",
    "106": "#1CF164",
    "107": "#0B34F5",
    "108": "#515CCD",
    "109": "#833F16",
    "110": "#11966B",
    "111": "#3BC321",
    "112": "#E87970",
    "113": "#DD3BA9",
    "114": "#B3E21F",
    "115": "#901983",
    "116": "#79A1F2",
    "117": "#E38A15",
    "118": "#0A33AE",
    "119": "#16E695",
    "120": "#8A20F4",
    "121": "#5B5CDE",
    "122": "#FA71E8",
    "123": "#1BFCEA",
    "124": "#47CA46",
    "125": "#86390A",
    "126": "#237888",
    "127": "#99B022",
    "128": "#76409A",
    "129": "#BE4D5B",
    "130": "#C3E759",
    "131": "#1B19DC",
    "132": "#B61AC7",
    "133": "#E369BD",
    "134": "#460384",
    "135": "#8858EB",
    "136": "#66CEFB",
    "137": "#1BAF95",
    "138": "#03CED9",
    "139": "#B2C10C",
    "140": "#8C366C",
    "141": "#41629A",
    "142": "#62A046",
    "143": "#B3F754",
    "144": "#CE2194",
    "145": "#7204CD",
    "146": "#2676E2",
    "147": "#A01F08",
    "148": "#04D665",
    "149": "#DA454E",
    "150": "#A16828",
    "151": "#391780",
    "152": "#6ABDFE",
    "153": "#0DF9B3",
    "154": "#FE2B02",
    "155": "#B0229D",
    "156": "#9BEC72",
    "157": "#933AE8",
    "158": "#48C83B",
    "159": "#F36AC5",
    "160": "#024BA9",
    "161": "#A3254D",
    "162": "#268D93",
    "163": "#87A80C",
    "164": "#34F8A3",
    "165": "#EB2D0C",
    "166": "#4A00FE",
    "167": "#60BCCF",
    "168": "#E1B000",
    "169": "#842CA0",
    "170": "#6550ED",
    "171": "#854F0C",
    "172": "#F08345",
    "173": "#39C36B",
    "174": "#FA16E9",
    "175": "#7BFE0D",
    "176": "#C75D83",
    "177": "#261090",
    "178": "#27ACC7",
    "179": "#1F07F0",
    "180": "#71B9FD",
    "181": "#A99C13",
    "182": "#29DA04",
    "183": "#713AD5",
    "184": "#0BAC50",
    "185": "#B13335",
    "186": "#65C64F",
    "187": "#FC0A0D",
    "188": "#D6639D",
    "189": "#A211A1",
    "190": "#2B529F",
    "191": "#48CCBA",
    "192": "#B7E83C",
    "193": "#0528E3",
    "194": "#E149F2",
    "195": "#2B9072",
    "196": "#84014C",
    "197": "#BF6D1E",
    "198": "#0394CA",
    "199": "#6EE96F",
    "200": "#EE1797",
    "201": "#5C2FAD",
    "202": "#91AF0E",
    "203": "#63A2F9",
    "204": "#1FFC47",
    "205": "#ADEB3D",
    "206": "#0B4680",
    "207": "#610CFB",
    "208": "#F57302",
    "209": "#057ACA",
    "210": "#6DF6B9",
    "211": "#449C63",
    "212": "#A42BD4",
    "213": "#1DD1D0",
    "214": "#891539",
    "215": "#6B9CEE",
    "216": "#D2B656",
    "217": "#9D8A09",
    "218": "#15DD2C",
    "219": "#22FC90",
    "220": "#6EF13F",
    "221": "#571A9E",
    "222": "#0E3DC0",
    "223": "#4CFCFB",
    "224": "#DF3D75",
    "225": "#8E55D7",
    "226": "#B83A21",
    "227": "#308E45",
    "228": "#35B29C",
    "229": "#00C5F2",
    "230": "#08DE16",
    "231": "#AA9739",
    "232": "#652E84",
    "233": "#1D0996",
    "234": "#62D535",
    "235": "#C85483",
    "236": "#3366B3",
    "237": "#9C4129",
    "238": "#63F2C7",
    "239": "#12915B",
    "240": "#F8A545",
    "241": "#24B4E8",
    "242": "#F35C1C",
    "243": "#11EA51",
    "244": "#900262",
    "245": "#A3CA48",
    "246": "#CCF602",
    "247": "#526BBF",
    "248": "#D01C8B",
    "249": "#58F307",
    "250": "#2A4582",
    "251": "#B78B12",
    "252": "#1D8118",
    "253": "#EEB574",
    "254": "#19918A",
    "255": "#6BFAB4",
    "256": "#731C80",
    "257": "#994207",
    "258": "#0F3EEE",
    "259": "#B00433",
    "260": "#D63D77",
    "261": "#6537D0",
    "262": "#1CE0B9",
    "263": "#BA961D",
    "264": "#49D94C",
    "265": "#F6CD73",
    "266": "#04C0FC",
    "267": "#80C904",
    "268": "#FBC710",
    "269": "#212E8E",
    "270": "#5EEDF4",
    "271": "#BC3FD0",
    "272": "#D73936",
    "273": "#11CD10",
    "274": "#7F5801",
    "275": "#66C55E",
    "276": "#A6007A",
    "277": "#3B8DA7",
    "278": "#F65B91",
    "279": "#D9D40C",
    "280": "#0259D3",
    "281": "#4924D7",
    "282": "#082783",
    "283": "#A48835",
    "284": "#BAE661",
    "285": "#64E1A7",
    "286": "#47CD41",
    "287": "#9A3C1F",
    "288": "#3D96AB",
    "289": "#BD4975",
    "290": "#EA24F7",
    "291": "#B97BFB",
    "292": "#FA1981",
    "293": "#202491",
    "294": "#588C09",
    "295": "#918D40",
    "296": "#55DCDE",
    "297": "#7A48FB",
    "298": "#2F59E9",
    "299": "#41AF73",
    "-1" : "#E1D9D1",
    "None" : "#E1D9D1"
}

In [ ]:
dataset_batch_palette = {new_key: value for new_key, value in zip(
    adata_full.obs['dataset_batch'].unique().tolist(),
    custom_palette_300.values())}

In [33]:
# Compute neighbors and UMAP 
sc.pp.neighbors(adata_full,
                n_neighbors=15,
                use_rep=f'neighborhood_emb_layer_{n_layer}',
                n_pcs=(n_pcs if n_pcs > 0 else None),
                key_added='neighborhood')
sc.pp.neighbors(adata_full,
                n_neighbors=15,
                use_rep=f'cell_emb_layer_{n_layer}',
                n_pcs=(n_pcs if n_pcs > 0 else None),
                key_added='cell')   

In [ ]:
sc.tl.umap(adata_full,
           neighbors_key='neighborhood',
           key_added='neighborhood_umap')

/nfs/users/nfs_s/sb75/.venvs/nichejepa/lib/python3.10/site-packages/umap/spectral.py:548: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


In [ ]:
sc.pl.umap(
    adata_full,
    neighbors_key='neighborhood',
    color=label,
    palette=(dataset_batch_palette if label == 'dataset_batch' else None),
    show=False)
plt.savefig(
    f'{artifacts_folder_path}/hst_corpus_80m/{args.load_timestamp}/{f"{args.dataset_name}_" if args.dataset_name != "hst_corpus_70m" else ""}{label}_neighborhood_umap.png',
    dpi=300,
    bbox_inches='tight')

In [ ]:
        sc.tl.umap(adata_full,
                   neighbors_key='cell',
                   key_added='cell_umap')

In [ ]:
        if emb_key == 'cell_emb_layer_6':
            del adata.obsm['neighborhood_emb_layer_6']
        elif emb_key == 'neighborhood_emb_layer_6':
            del adata.obsm['cell_emb_layer_6']

In [ ]:
!free -h

In [ ]:
import time
from datetime import timedelta

start_time = time.time()
print("Start time:", start_time)

# Compute neighbors and UMAP 
sc.pp.neighbors(adata_full,
                n_neighbors=15,
                use_rep=emb_key,
                n_pcs=50,
                key_added='neighborhood')

intermediate_time = time.time()
print("Intermediate time:", intermediate_time)

elapsed_time = intermediate_time - start_time
print("Elapsed time:", str(timedelta(seconds=elapsed_time)))

sc.tl.umap(adata,
           neighbors_key='neighborhood')

end_time = time.time()
print("End time:", end_time)

elapsed_time = end_time - start_time
print("Elapsed time:", str(timedelta(seconds=elapsed_time)))

## 2.3 Analysis Data

### 2.3.1 Kidney Only

#### 2.3.1 Run Model Inference

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
                 '1007_batch0',
                 '1007_batch1',
                 '1007_batch2',
                 '1007_batch3',
                 '1007_batch4',
                 '1007_batch5',
                 '1007_batch6',
                 '1007_batch7',
                 '1007_batch8',
                 '1007_batch9',
                 '1007_batch10',
                 '1007_batch11',
                 '1007_batch12',
                 '1007_batch13',
                 '1007_batch14',
                 '1007_batch15',
                 '1007_batch16',
                 '1007_batch17',
                 '1007_batch18',
                 '1007_batch19',
                 '1007_batch20',
                 '1007_batch21',
                 '1007_batch22',
                 '1007_batch23',
                 '1007_batch24',
                 '1007_batch25',
                 '1007_batch26',
                 '1007_batch27',
                 '1007_batch28',
                 '1007_batch29',
                 '1007_batch30',
                 '1007_batch31',
                 '1007_batch32',
                 '1007_batch33',
                 '1007_batch34',
                 '1007_batch35',
                 '1007_batch36',
                 '1007_batch37',
                 '1007_batch38',
                 '1007_batch39',
                 '1007_batch40',]
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Load data
_, _, dataset = prepare_dataset(params)
dataset

In [ ]:
# Subset dataset
step_size = 350000
steps = int(np.ceil(len(dataset)/step_size))

dataset_subsets = []

for i in range(steps):
    if ((i+1)*step_size) > len(dataset):
        end_step = len(dataset)
    else:
        end_step = (i+1)*step_size
    dataset_subsets.append(dataset.select(np.arange(i*step_size, end_step)))

del dataset

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    print(f'Iteration: {i}')
    #if i != 1:
    #    continue
    
    # Get dataset IDs
    dataset_ids = ['1007']
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_1007_{step_size}_{i}.h5ad')
    del adata

#### 2.3.2 Combine Data

In [ ]:
load_timestamp = '11062025_183320_055'
step_size = '350000'
read_checkpoint = 'ep10_0'
emb_key = 'neighborhood_emb_layer_6'

In [ ]:
adata_list = []

for i in range(8):
    adata = sc.read_h5ad(f'{artifacts_folder_path}/hst_corpus_80m/{load_timestamp}/adata_{read_checkpoint}_1007_{step_size}_{i}.h5ad')
    adata_list.append(adata)
    del adata

adata_full = sc.concat(adata_list, join='outer', index_unique=None)

In [ ]:
adata_full.write('adata_ep10_0_1007_complete.h5ad')

In [ ]:
!free -h

### 2.3.2 Joint Skin, Kidney, Embryo

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [
                 '1007_batch0',
                 '1007_batch1',
                 '1007_batch2',
                 '1007_batch3',
                 '1007_batch4',
                 '1007_batch5',
                 '1007_batch6',
                 '1007_batch7',
                 '1007_batch8',
                 '1007_batch9',
                 '1007_batch10',
                 '1007_batch11',
                 '1007_batch12',
                 '1007_batch13',
                 '1007_batch14',
                 '1007_batch15',
                 '1007_batch16',
                 '1007_batch17',
                 '1007_batch18',
                 '1007_batch19',
                 '1007_batch20',
                 '1007_batch21',
                 '1007_batch22',
                 '1007_batch23',
                 '1007_batch24',
                 '1007_batch25',
                 '1007_batch26',
                 '1007_batch27',
                 '1007_batch28',
                 '1007_batch29',
                 '1007_batch30',
                 '1007_batch31',
                 '1007_batch32',
                 '1007_batch33',
                 '1007_batch34',
                 '1007_batch35',
                 '1007_batch36',
                 '1007_batch37',
                 '1007_batch38',
                 '1007_batch39',
                 '1007_batch40',]
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

In [ ]:
# Load data
_, _, dataset = prepare_dataset(params)
dataset

In [ ]:
dataset

In [ ]:
# Subset dataset
step_size = 350000
steps = int(np.ceil(len(dataset)/step_size))

dataset_subsets = []

for i in range(steps):
    if ((i+1)*step_size) > len(dataset):
        end_step = len(dataset)
    else:
        end_step = (i+1)*step_size
    dataset_subsets.append(dataset.select(np.arange(i*step_size, end_step)))

del dataset

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    print(f'Iteration: {i}')
    if i != 0:
        continue
    
    # Get dataset IDs
    dataset_ids = ['1007']
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_1007.h5ad')
    del adata

### 2.3.2 GBM

In [ ]:
# Load token dict
with open(token_dict_path, 'rb') as f:
    token_dict = pickle.load(f)

token_dict_reverse =  {value: key for key, value in token_dict.items()}

In [ ]:
# Set test batch IDs
params['data']['precomputed_split'] = False
params['data']['test_batch_ids'] = [f'1002_batch{i}' for i in range(75)]
params['data']['val_batch_ids'] = None

In [ ]:
# Load data
_, _, dataset = prepare_dataset(params)
dataset

In [ ]:
# Subset dataset
step_size = 2000000
steps = int(np.ceil(len(dataset)/step_size))

dataset_subsets = []

for i in range(steps):
    if ((i+1)*step_size) > len(dataset):
        end_step = len(dataset)
    else:
        end_step = (i+1)*step_size
    dataset_subsets.append(dataset.select(np.arange(i*step_size, end_step)))

del dataset

In [ ]:
len(dataset_subsets)

In [ ]:
from pathlib import Path
folder_path = Path(f'/lustre/scratch126/cellgen/team361/DATASETS/silver/xhb1002-AT9-6b_1p')
adata_path_list = [p for p in folder_path.rglob("*")]

for adata_path in adata_path_list:
    adata = sc.read_h5ad(adata_path)
    print(f'{adata_path}: {adata.uns["batch"]}')

In [ ]:
# Get batches for current dataset subset
for i, dataset_subset in enumerate(dataset_subsets):
    if i < 8:
        continue
    print(f'Iteration: {i}')
    
    # Get batches
    batches = list(set([token_dict_reverse[batch_value[0]] for batch_value in dataset_subset['batch_value_token']]))
    print(f'Batches: {batches}')
    
    # Get dataset IDs
    dataset_ids = list(set([batch.split('_')[1] for batch in batches]))
    print(f'Dataset IDs: {dataset_ids}')

    # Run model inference
    adata = infer(params,
                  dataset_subset,
                  load_folder_path=folder_path,
                  dataset_ids=dataset_ids,
                  obs_cols=None,
                  uns_cols=['tissue'],
                  agg_type=agg_type,
                  masked_tokens=None,
                  agg_excluded_tokens=None,
                  cell_gene_ids=[],
                  neighborhood_gene_ids=[]
                 )
    
    # Save adata
    adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_1002_{step_size}_{i}.h5ad')
    del adata

In [ ]:
# Run model inference
adata = infer(params,
              test_dataset,
              load_folder_path=folder_path,
              dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
              agg_type=agg_type,
              masked_tokens=None,
              agg_excluded_tokens=None,
              cell_gene_ids=[],
              neighborhood_gene_ids=[])

# Save adata
# adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

In [ ]:
# Set inference params
layer = params['meta']['enc_depth']
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
latent_leiden_resolution = 0.5 # 0.1, 0.5
latent_cluster_key = f'leiden_res{str(latent_leiden_resolution).replace(".", "_")}'

In [ ]:
item = dataset[0]

In [ ]:
item['cell_id']

In [ ]:
dataset_subset = dataset.filter(lambda x: x['cell_id'].split('_')[0] in batch_ids)

In [ ]:
dataset_subset

In [ ]:
params['data']['tokenized_data_folder_path']

In [ ]:
test_subset = test_dataset.select(range(10))
test_subset
gene_ids = list(np.unique(test_subset['gene_tokens']))

# Run model inference
adata = infer(params,
              test_subset,
              load_folder_path=folder_path,
              dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
              obs_cols=[niche_key],
              agg_type='avg',
              masked_tokens=None,
              agg_excluded_tokens=None,
              cell_gene_ids=[],
              neighborhood_gene_ids=gene_ids)

# Save adata
#adata.write(f'{folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

norms = []

for i in range(10):
    for key in adata.obsm.keys():
        if 'neighborhood_emb_gene' in key:
            gene_emb = adata.obsm[key][i]
            norm = np.linalg.norm(gene_emb)
            norms.append(norm)

filtered_data = [x for x in norms if x != 0]

# Plot histogram of filtered data
plt.hist(filtered_data, bins=5, edgecolor='black')

## 2.4 Miscellaneous

In [ ]:
# Check batches
from pathlib import Path
folder_path = Path(f'/lustre/scratch126/cellgen/team361/DATASETS/silver/xhb1002-AT9-6b_1p')
adata_path_list = [p for p in folder_path.rglob("*")]

for adata_path in adata_path_list:
    adata = sc.read_h5ad(adata_path)
    print(f'{adata_path}: {adata.uns["batch"]}')

In [ ]:
adata_batch = adata[adata.obs['slide_id'] == 'B2']

sc.pl.spatial(adata_batch,
              color='subregion',
              spot_size=30)

In [ ]:
sc.pp.calculate_qc_metrics(
    adata,
    inplace=True
)

import seaborn as sns
import matplotlib.pyplot as plt
sns.histplot(data=adata.obs, x='n_genes_by_counts', bins=100)
plt.xlabel('Number of expressed genes per cell')
plt.ylabel('Number of cells')
plt.show()

In [ ]:
# Pickle the list
with open("human_cohort2_60m_32_gene_corrected_read_depth_nonzero_mean_None_gene_corrected_read_depth_None_shifted_log_knn_10_train.pkl", "rb") as f:
    train = pickle.load(f)

with open("human_cohort2_60m_32_gene_corrected_read_depth_nonzero_mean_None_gene_corrected_read_depth_None_shifted_log_knn_10_test.pkl", "rb") as f:
    test = pickle.load(f)

with open("human_cohort2_60m_32_None_None_None_gene_corrected_read_depth_None_shifted_log_knn_10_train.pkl", "rb") as f:
    new_train = pickle.load(f)

with open("human_cohort2_60m_32_None_None_None_gene_corrected_read_depth_None_shifted_log_knn_10_test.pkl", "rb") as f:
    new_test = pickle.load(f)

print(len(train))
print(len(test))
print(len(new_train))
print(len(new_test))

train_matches = set(train).intersection(new_train)
print(len(train_matches))
test_matches = set(test).intersection(new_test)
print(len(test_matches))

train_differences = []
for i in range(len(train)):
    if train[i] != new_train[i]:
        train_differences.append((i, train[i], new_train[i]))
print(train_differences)
print(len(differences))

test_differences = []
for i in range(len(test)):
    if test[i] != new_test[i]:
        test_differences.append((i, test[i], new_test[i]))
print(test_differences[-10:])